In [0]:
# ============================================================
# 01_raw_to_bronze_chicago
# ============================================================
# Purpose  : Read Chicago CSV from Volume 
# Bronze 
#   inferSchema=false  - all 17 columns land as STRING
#   No filtering       - every source row is preserved
#   No type casting    - Silver handles all type conversion
#   No cleaning        - raw data must be exactly as received 
#   APPEND mode        - every run adds a new batch, never overwrites


In [0]:
import uuid
from pyspark.sql import functions as F

CATALOG = "final_project_databi_group8"

CHI_PATH     = "/Volumes/final_project_databi_group8/default/finalprojectgroup8volume/Food_Inspections_20260408.csv"
BRONZE_TABLE = f"{CATALOG}.bronze.chicago_raw"

BATCH_ID = str(uuid.uuid4())
RUN_ID   = "01_raw_to_bronze_chicago"
SCHEMA_V = "v1"

print(f"Catalog      : {CATALOG}")
print(f"Source file  : {CHI_PATH}")
print(f"Target table : {BRONZE_TABLE}")
print(f"Batch ID     : {BATCH_ID}")
print(f"Run ID       : {RUN_ID}")

In [0]:
# Replace this cell entirely
files = [f.name for f in dbutils.fs.ls("/Volumes/final_project_databi_group8/default/finalprojectgroup8volume/")]
print("Files in Volume:")
for f in files:
    print(f"  {f}")

assert "Food_Inspections_20260408.csv" in files, "ERROR: Chicago CSV not found."
print("\n Chicago CSV confirmed")

In [0]:
# Read raw CSV 
# Bronze preserves the source exactly. All 17 columns arrive as STRING.
# multiLine=true handles violation text that spans multiple lines.
df_chi_raw = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "false")
    .option("multiLine",   "true")
    .option("escape",      '"')
    .option("encoding",    "UTF-8")
    .csv(CHI_PATH)
)

print(f"Row count : {df_chi_raw.count():,}")
print(f"Col count : {len(df_chi_raw.columns)}")

In [0]:
# Rename 17 columns to snake_case because delta Lake rejects column names with spaces and special characters.
df_chi = (
    df_chi_raw
    .withColumnRenamed("Inspection ID",   "inspection_id")
    .withColumnRenamed("DBA Name",        "restaurant_name")
    .withColumnRenamed("AKA Name",        "aka_name")
    .withColumnRenamed("License #",       "license_number")
    .withColumnRenamed("Facility Type",   "facility_type")
    .withColumnRenamed("Risk",            "risk_category")
    .withColumnRenamed("Address",         "street_address")
    .withColumnRenamed("City",            "city")
    .withColumnRenamed("State",           "state")
    .withColumnRenamed("Zip",             "zip_code")
    .withColumnRenamed("Inspection Date", "inspection_date")
    .withColumnRenamed("Inspection Type", "inspection_type")
    .withColumnRenamed("Results",         "inspection_result")
    .withColumnRenamed("Violations",      "violations_raw")
    .withColumnRenamed("Latitude",        "latitude")
    .withColumnRenamed("Longitude",       "longitude")
    .withColumnRenamed("Location",        "location_raw")
)

# ── Add audit metadata columns
# _extract_ts uses F.current_timestamp() - typed as TIMESTAMP.

df_chi = (
    df_chi
    .withColumn("city_source",      F.lit("CHI"))
    .withColumn("_batch_id",        F.lit(BATCH_ID))
    .withColumn("_extract_ts",      F.current_timestamp())
    .withColumn("_extract_date",    F.current_date().cast("string"))
    .withColumn("_source_file",     F.lit("Food_Inspections_20260408.csv"))
    .withColumn("_run_id",          F.lit(RUN_ID))
    .withColumn("_schema_version",  F.lit(SCHEMA_V))
    .withColumn("_row_hash",        F.sha2(
        F.concat_ws("||",
            F.coalesce(F.col("inspection_id"),   F.lit("")),
            F.coalesce(F.col("violations_raw"),  F.lit("")),
            F.coalesce(F.col("inspection_date"), F.lit(""))
        ), 256))
)

# Verify no special characters remain in column names
bad_cols = [c for c in df_chi.columns
            if any(x in c for x in [" ", "-", "(", ")", ","])]
if bad_cols:
    raise ValueError(f"Bad column names: {bad_cols}")

print(f"All {len(df_chi.columns)} column names are Delta-safe")

display(df_chi.select(
    "inspection_id", "restaurant_name", "inspection_result",
    "city_source", "_batch_id", "_extract_ts", "_row_hash"
).limit(3))


In [0]:
# Write to Bronze - APPEND mode 
# Every run produces a new batch (different _batch_id).
# Silver always reads only the latest batch using _batch_id.
# Delta version history records every append as a new version.
(
    df_chi
    .write
    .format("delta")
    .mode("append")
    .partitionBy("city_source", "_extract_date")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

written = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == BATCH_ID)
    .count()
)
source = df_chi.count()

print(f"  Written to  : {BRONZE_TABLE}")
print(f"  Batch ID    : {BATCH_ID}")
print(f"  Source rows : {source:,}")
print(f"  Written rows: {written:,}")
print(f"  Match       : {'OK' if source == written else 'MISMATCH'}")

assert source == written, f"Mismatch: source={source}, written={written}"


In [0]:
# history 
print("All batches ever loaded into bronze.chicago_raw:")
display(spark.sql(f"""
    SELECT
        city_source,
        _extract_date,
        _extract_ts,
        LEFT(_batch_id, 8)  AS batch_short,
        _run_id,
        _schema_version,
        COUNT(*)            AS row_count
    FROM {BRONZE_TABLE}
    GROUP BY city_source, _extract_date, _extract_ts,
             _batch_id, _run_id, _schema_version
    ORDER BY _extract_ts DESC
"""))

print("\nDelta version history (one version per append run):")
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}"))
